In [ ]:
import json, os, itertools, statistics, math
from pathlib import Path
from collections import Counter, defaultdict

import torch
from datasets import load_dataset
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

# Local model helpers (ensure these .py files are on PYTHONPATH)
from GinSign import BERT_Lifter
from GinSign import BERT_Grounder

from transformers import T5ForConditionalGeneration, T5TokenizerFast

# ---- Path configuration ----
ROOT = Path.cwd()                           # run notebook from repo root
LIFT_TEST_DIR      = ROOT / "raw_tl_data" / "VLTL-Bench" / "lifting"   / "test"
GROUND_TEST_DIR    = ROOT / "raw_tl_data" / "VLTL-Bench" / "grounding" / "test"
RAW_TL_TEST_DIR    = ROOT / "raw_tl_data" / "VLTL-Bench" / "test"
FINE_GROUND_TEST_DIR    = ROOT / "fine_prefix_grounding" / "test"

LIFTER_MODEL_DIR   = ROOT / "BERT_Lifter_Demo"
GROUND_MODEL_DIR   = ROOT / "BERT_Grounding_Model"
T5_MODEL_DIR       = ROOT / "translation_models" / "t5-base_grounded_sentence_masked_tl"
GRAFT_MODEL_DIR       = ROOT / "GraFT" / "translation_models" / "t5-base_grounded_sentence_masked_tl"

LIFT_OUT_DIR       = ROOT / "lifting_evaluations" / "GinSign"
GROUND_OUT_DIR     = ROOT / "grounding_evaluations"
TRANS_OUT_DIR      = ROOT / "translation_evaluations"

for d in (LIFT_OUT_DIR, GROUND_OUT_DIR, TRANS_OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

def metrics(true, pred):
    pr, re, f1, _ = precision_recall_fscore_support(true, pred, average='macro', zero_division=0)
    acc = accuracy_score(true, pred)
    return {'accuracy': acc, 'precision': pr, 'recall': re, 'f1': f1}

def print_report(report_dict, title):
    from IPython.display import display, Markdown
    md = [f"### {title}", "|domain|acc|prec|rec|f1|", "|---|---|---|---|---|"]
    for dom, met in report_dict.items():
        md.append(f"|{dom}|{met['accuracy']:.3f}|{met['precision']:.3f}|{met['recall']:.3f}|{met['f1']:.3f}|")
    display(Markdown("n".join(md)))


def labels_to_propdict(words, labels):
    """words & labels are same length. Returns {'prop_1': [...], ...}"""
    out, cur_lab, cur_tokens = {}, None, []
    for tok, lab in zip(words, labels):
        if lab == cur_lab:                # keep accumulating
            if lab != 0:
                cur_tokens.append(tok)
        else:                             # boundary
            if cur_lab not in (None, 0):  # flush previous chunk
                out[f"prop_{cur_lab}"] = cur_tokens
            cur_lab  = lab
            cur_tokens = [tok] if lab != 0 else []
    # flush tail
    if cur_lab not in (None, 0):
        out[f"prop_{cur_lab}"] = cur_tokens
    return out
lifter = BERT_Lifter(LIFTER_MODEL_DIR)
lift_metrics = {}

for fp in LIFT_TEST_DIR.glob("*.jsonl"):
    domain = fp.stem
    true_labels = []
    pred_labels = []
    out_rows   = []

    for row in load_dataset("json", data_files=str(fp), split="train"):
        words     = row["sentence"]
        true_lab  = row["lifted_sentence_prop_ids"]
        pred_dict = lifter.lift(" ".join(words))

        # build predicted label list
        pred_lab = [0]*len(words)
        for k, toks in pred_dict.items():
            idx = int(k.split("_")[1])
            for i, tok in enumerate(words):
                if tok in toks:
                    pred_lab[i] = idx

        # ---- new: compact prop_dict from contiguous spans ----
        pred_propdict = labels_to_propdict(words, pred_lab)

        row_out = dict(row)
        row_out["predicted_labels"]   = pred_lab
        row_out["predicted_propdict"] = pred_propdict
        out_rows.append(row_out)

    # Save predictions
    with (LIFT_OUT_DIR / f"{domain}.jsonl").open("w") as f:
        for r in out_rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    lift_metrics[domain] = metrics(true_labels, pred_labels)

print_report(lift_metrics, "Lifting Metrics")

In [ ]:
lifter = BERT_Lifter(LIFTER_MODEL_DIR)
lift_metrics = {}

for fp in LIFT_TEST_DIR.glob("*.jsonl"):
    domain = fp.stem
    true_labels = []
    pred_labels = []
    out_rows   = []

    for row in load_dataset("json", data_files=str(fp), split="train"):
        words     = row["sentence"]
        true_lab  = row["lifted_sentence_prop_ids"]
        pred_dict = lifter.lift(" ".join(words))

        # build predicted label list
        pred_lab = [0]*len(words)
        for k, toks in pred_dict.items():
            idx = int(k.split("_")[1])
            for i, tok in enumerate(words):
                if tok in toks:
                    pred_lab[i] = idx

        # ---- new: compact prop_dict from contiguous spans ----
        pred_propdict = labels_to_propdict(words, pred_lab)

        row_out = dict(row)
        row_out["predicted_labels"]   = pred_lab
        row_out["predicted_propdict"] = pred_propdict
        out_rows.append(row_out)

    # Save predictions
    with (LIFT_OUT_DIR / f"{domain}.jsonl").open("w") as f:
        for r in out_rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    lift_metrics[domain] = metrics(true_labels, pred_labels)

print_report(lift_metrics, "Lifting Metrics")

In [ ]:
GROUND_MODEL_DIR   = ROOT / "BERT_Grounding_Model_sr+tl"
GROUND_OUT_DIR     = ROOT / "grounding_evaluations" / "sr+tl"

grounder = BERT_Grounder(GROUND_MODEL_DIR)
ground_metrics = {}

for fp in GROUND_TEST_DIR.glob("*.jsonl"):
    domain = fp.stem
    true_all, pred_all = [], []
    out_rows = []

    for row in load_dataset("json", data_files=str(fp), split="train"):

        prefix = row["prefix"]
        sent   = row["sentence"]
        true   = row["prefix_target"]
        pred   = grounder.ground(sent, prefix)

        # print(f"True: {true}")
        # print(f"Pred: {pred}")
        # print(sent)
        # print()
        true_all.extend(true)
        pred_all.extend(pred)

        r = dict(row)
        r["predicted_prefix_target"] = pred
        out_rows.append(r)

    with (GROUND_OUT_DIR / f"{domain}.jsonl").open("w") as f:
        for r in out_rows:
            f.write(json.dumps(r, ensure_ascii=False)+"\n")    

    ground_metrics[domain] = metrics(true_all, pred_all)

print_report(ground_metrics, "Grounding Metrics")

In [ ]:
GROUND_MODEL_DIR   = ROOT / "BERT_Grounding_Model_wh+sr"
GROUND_OUT_DIR     = ROOT / "grounding_evaluations" / "wh+sr"

grounder = BERT_Grounder(GROUND_MODEL_DIR)
ground_metrics = {}

for fp in GROUND_TEST_DIR.glob("*.jsonl"):
    domain = fp.stem
    true_all, pred_all = [], []
    out_rows = []

    for row in load_dataset("json", data_files=str(fp), split="train"):

        prefix = row["prefix"]
        sent   = row["sentence"]
        true   = row["prefix_target"]
        pred   = grounder.ground(sent, prefix)

        # print(f"True: {true}")
        # print(f"Pred: {pred}")
        # print(sent)
        # print()
        true_all.extend(true)
        pred_all.extend(pred)

        r = dict(row)
        r["predicted_prefix_target"] = pred
        out_rows.append(r)

    with (GROUND_OUT_DIR / f"{domain}.jsonl").open("w") as f:
        for r in out_rows:
            f.write(json.dumps(r, ensure_ascii=False)+"\n")    

    ground_metrics[domain] = metrics(true_all, pred_all)

print_report(ground_metrics, "Grounding Metrics")

In [ ]:
GROUND_MODEL_DIR   = ROOT / "BERT_Grounding_Model_wh+tl"
GROUND_OUT_DIR     = ROOT / "grounding_evaluations" / "wh+tl"

grounder = BERT_Grounder(GROUND_MODEL_DIR)
ground_metrics = {}

for fp in GROUND_TEST_DIR.glob("*.jsonl"):
    domain = fp.stem
    true_all, pred_all = [], []
    out_rows = []

    for row in load_dataset("json", data_files=str(fp), split="train"):

        prefix = row["prefix"]
        sent   = row["sentence"]
        true   = row["prefix_target"]
        pred   = grounder.ground(sent, prefix)

        # print(f"True: {true}")
        # print(f"Pred: {pred}")
        # print(sent)
        # print()
        true_all.extend(true)
        pred_all.extend(pred)

        r = dict(row)
        r["predicted_prefix_target"] = pred
        out_rows.append(r)

    with (GROUND_OUT_DIR / f"{domain}.jsonl").open("w") as f:
        for r in out_rows:
            f.write(json.dumps(r, ensure_ascii=False)+"\n")    

    ground_metrics[domain] = metrics(true_all, pred_all)

print_report(ground_metrics, "Grounding Metrics")

In [ ]:
GROUND_MODEL_DIR   = ROOT / "BERT_Grounding_Model"
GROUND_OUT_DIR     = ROOT / "grounding_evaluations" / "full"

grounder = BERT_Grounder(GROUND_MODEL_DIR)
ground_metrics = {}

for fp in GROUND_TEST_DIR.glob("*.jsonl"):
    domain = fp.stem
    true_all, pred_all = [], []
    out_rows = []

    for row in load_dataset("json", data_files=str(fp), split="train"):

        prefix = row["prefix"]
        sent   = row["sentence"]
        true   = row["prefix_target"]
        pred   = grounder.ground(sent, prefix)

        # print(f"True: {true}")
        # print(f"Pred: {pred}")
        # print(sent)
        # print()
        true_all.extend(true)
        pred_all.extend(pred)

        r = dict(row)
        r["predicted_prefix_target"] = pred
        out_rows.append(r)

    with (GROUND_OUT_DIR / f"{domain}.jsonl").open("w") as f:
        for r in out_rows:
            f.write(json.dumps(r, ensure_ascii=False)+"\n")    

    ground_metrics[domain] = metrics(true_all, pred_all)

print_report(ground_metrics, "Grounding Metrics")

In [ ]:


t5_tok = T5TokenizerFast.from_pretrained(T5_MODEL_DIR)
t5_mod = T5ForConditionalGeneration.from_pretrained(T5_MODEL_DIR).eval()
t5_mod.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))

PREFIX_SENT = "Translate the following sentence into Linear Temporal Logic: "

def make_grounded_sentence(words, pred_labels):
    out = []
    for tok, lab in zip(words, pred_labels):
        if lab == 0:
            out.append(tok)
        else:
            if len(out)>0 and out[-1] == "prop_"+str(lab):
                continue
            out.append(f"prop_{lab}")
    return out

trans_metrics = {}
for fp in RAW_TL_TEST_DIR.glob("*.jsonl"):
    domain = fp.stem
    out_rows = []
    exact_matches = 0
    total = 0

    # Load previously saved lift & ground predictions
    lift_path = LIFT_OUT_DIR / f"{domain}.jsonl"
    lift_pred_map   = {}
    lift_prop_map   = {}
    with open(lift_path, encoding="utf‑8") as f:
        for line in f:
            obj = json.loads(line)
            lift_pred_map[obj["id"]]  = obj["predicted_labels"]
            lift_prop_map[obj["id"]]  = obj["predicted_propdict"]

    for row in load_dataset("json", data_files=str(fp), split="train"):

        words = row["sentence"]
        row_id = str(row["id"])

        pred_lab = lift_pred_map.get(row_id, row["lifted_sentence_prop_ids"])
        grounded_sent = make_grounded_sentence(words, pred_lab)
        print(grounded_sent)
        enc = t5_tok([PREFIX_SENT + " ".join(grounded_sent)], return_tensors="pt", truncation=True).to(t5_mod.device)
        gen_ids = t5_mod.generate(**enc, max_length=128)
        pred_masked = t5_tok.batch_decode(gen_ids, skip_special_tokens=True)[0].split()

        gold = " ".join(row["masked_tl"])
        pred = " ".join(pred_masked)

        exact_matches += int(pred == gold)
        total += 1

        row_out = dict(row)
        row_out["predicted_masked_tl"] = pred_masked
        out_rows.append(row_out)

    with (TRANS_OUT_DIR / f"{domain}.jsonl").open("w") as f:
        for r in out_rows:
            f.write(json.dumps(r, ensure_ascii=False)+ "\n")

    trans_metrics[domain] = {"exact_match": exact_matches/total}

from IPython.display import display, Markdown
md = ["### Translation Exact‑Match Accuracy", "|domain|acc|", "|---|---|"]
for d,m in trans_metrics.items():
    md.append(f"|{d}|{m['exact_match']:.3f}|")
display(Markdown("\n".join(md)))

In [ ]:
from utils.TemporalLogicGrammar import TemporalLogicGrammar
from utils.TemporalLogitsProcessor_Inference import TemporalLogitsProcessorInference
from transformers import T5Tokenizer

t5_tok = T5Tokenizer.from_pretrained(GRAFT_MODEL_DIR)
t5_mod = T5ForConditionalGeneration.from_pretrained(GRAFT_MODEL_DIR).eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Make sure generation_config is set
t5_mod.config.decoder_start_token_id = t5_tok.pad_token_id
t5_mod.generation_config._from_model_config = True
t5_mod.generation_config.decoder_start_token_id = t5_tok.pad_token_id
t5_mod.generation_config.eos_token_id = t5_tok.eos_token_id
t5_mod.generation_config.pad_token_id = t5_tok.pad_token_id

# Prepare a restricted set of valid token IDs (if needed)
    # Define valid temporal logic tokens
temporal_operators = ["finally", "globally", "until"]
logical_operators = ["and", "or", "implies", "double_implies", "not", "(", ")"]
propositions = ["prop_", "_1", "_2", "_3", "_4", "_5"]
valid_tokens_str = temporal_operators + logical_operators + propositions
valid_token_ids = set()
for token in valid_tokens_str:
    token_ids = t5_tok.encode(token, add_special_tokens=False)
    print(f'Token: {token}, TokenIDs: {token_ids}')
    valid_token_ids.update(token_ids)
valid_token_ids.add(t5_tok.pad_token_id)
valid_token_ids.add(t5_tok.eos_token_id)
valid_token_ids = list(valid_token_ids)
# Load grammar
grammar = TemporalLogicGrammar(t5_tok)

t5_mod.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
logits_processor = TemporalLogitsProcessorInference(
    grammar,
    valid_token_ids,
    t5_tok,
    t5_mod
)
PREFIX_SENT = "Translate the following sentence into Linear Temporal Logic: "

def make_grounded_sentence(words, pred_labels):
    out = []
    for tok, lab in zip(words, pred_labels):
        if lab == 0:
            out.append(tok)
        else:
            if len(out)>0 and out[-1] == "prop_"+str(lab):
                continue
            out.append(f"prop_{lab}")
    return out

trans_metrics = {}
for fp in RAW_TL_TEST_DIR.glob("*.jsonl"):
    domain = fp.stem
    out_rows = []
    exact_matches = 0
    total = 0

    # Load previously saved lift & ground predictions
    lift_path = LIFT_OUT_DIR / f"{domain}.jsonl"
    lift_pred_map   = {}
    lift_prop_map   = {}
    with open(lift_path, encoding="utf‑8") as f:
        for line in f:
            obj = json.loads(line)
            lift_pred_map[obj["id"]]  = obj["predicted_labels"]
            lift_prop_map[obj["id"]]  = obj["predicted_propdict"]

    for row in load_dataset("json", data_files=str(fp), split="train"):

        words = row["sentence"]
        row_id = str(row["id"])

        pred_lab = lift_pred_map.get(row_id, row["lifted_sentence_prop_ids"])
        grounded_sent = make_grounded_sentence(words, pred_lab)
        # print(grounded_sent)
        enc = t5_tok([PREFIX_SENT + " ".join(grounded_sent)], return_tensors="pt", truncation=True).to(t5_mod.device)
        gen_ids = t5_mod.generate(**enc, max_length=128, logits_processor=[logits_processor])
        pred_masked = t5_tok.batch_decode(gen_ids, skip_special_tokens=True)[0].split()

        gold = " ".join(row["masked_tl"])
        pred = " ".join(pred_masked)

        exact_matches += int(pred == gold)
        total += 1
        print(f"Gold: {gold}")
        print(f"Pred: {pred}")
        row_out = dict(row)
        row_out["predicted_masked_tl"] = pred_masked
        out_rows.append(row_out)

    with (TRANS_OUT_DIR / f"{domain}.jsonl").open("w") as f:
        for r in out_rows:
            f.write(json.dumps(r, ensure_ascii=False)+ "\n")

    trans_metrics[domain] = {"exact_match": exact_matches/total}

from IPython.display import display, Markdown
md = ["### Translation Exact‑Match Accuracy", "|domain|acc|", "|---|---|"]
for d,m in trans_metrics.items():
    md.append(f"|{d}|{m['exact_match']:.3f}|")
display(Markdown("\n".join(md)))

In [ ]:
import json
from pathlib import Path

DATA_DIR = Path("/home/will.english/Desktop/Research/AAAI_2026_GinSign/raw_tl_data/VLTL-Bench/total")

def count_unique_masked_tl(data_dir):
    for jsonl_file in sorted(data_dir.glob("*.jsonl")):
        unique_masked_tl = set()
        with open(jsonl_file, "r", encoding="utf-8") as f:
            for line in f:
                obj = json.loads(line)
                # Convert list to tuple for hashing
                masked_tl = tuple(obj.get("masked_tl", []))
                unique_masked_tl.add(masked_tl)
        print(f"{jsonl_file.name}: {len(unique_masked_tl)} unique 'masked_tl' entries")

count_unique_masked_tl(DATA_DIR)


In [ ]:
import json
from pathlib import Path
from collections import Counter

DATA_DIR = Path("/home/will.english/Desktop/Research/AAAI_2026_GinSign/raw_tl_data/VLTL-Bench/total_v2")

def analyze_masked_tl(data_dir):
    for jsonl_file in sorted(data_dir.glob("*.jsonl")):
        unique_masked_tl = set()
        operator_counts = Counter()

        with open(jsonl_file, "r", encoding="utf-8") as f:
            for line in f:
                obj = json.loads(line)
                masked_tl = obj.get("masked_tl", [])
                unique_masked_tl.add(tuple(masked_tl))
                operator_counts.update(masked_tl)

        print(f"\n{jsonl_file.name}:")
        print(f"  Unique 'masked_tl' sequences: {len(unique_masked_tl)}")
        print("  Operator/token frequencies:")
        for op, count in sorted(operator_counts.items()):
            print(f"    {op}: {count}")

analyze_masked_tl(DATA_DIR)


In [ ]:
#!/usr/bin/env python3
import json
from pathlib import Path

def extract_unique_props(data_dir):
    data_dir = Path(data_dir)
    for file_path in sorted(data_dir.glob("*.jsonl")):
        actions = set()
        args = set()
        with file_path.open("r", encoding="utf-8") as f:
            for line in f:
                obj = json.loads(line)
                for details in obj.get("prop_dict", {}).values():
                    # collect action_canon
                    action = details.get("action_canon")
                    if action is not None:
                        actions.add(action)
                    # collect each element of args_canon
                    for arg in details.get("args_canon", []):
                        args.add(arg)

        print(f"\n=== {file_path.name} ===")
        print("Unique action_canon values:")
        for a in sorted(actions):
            print(f"  - {a!r}")

        print("Unique args_canon elements:")
        for arg in sorted(args):
            print(f"  - {arg!r}")

if __name__ == "__main__":
    extract_unique_props("/home/will.english/Desktop/Research/AAAI_2026_GinSign/raw_tl_data/VLTL-Bench/total")


In [ ]:
#!/usr/bin/env python3
import json
from pathlib import Path
from transformers import BertTokenizerFast

def compute_bert_lengths(data_dir, bert_model="bert-base-uncased"):
    # load BERT tokenizer
    tokenizer = BertTokenizerFast.from_pretrained(bert_model)
    data_dir = Path(data_dir)

    for file_path in sorted(data_dir.glob("*.jsonl")):
        actions = set()
        args = set()

        # collect unique action_canon and normalized args_canon elements
        with file_path.open("r", encoding="utf-8") as f:
            for line in f:
                obj = json.loads(line)
                for details in obj.get("prop_dict", {}).values():
                    # action canon
                    action = details.get("action_canon")
                    if action:
                        actions.add(action)

                    # args canon, but normalize any ending in "street"/"avenue" to "road"
                    for arg in details.get("args_canon", []):
                        arg_lower = arg.lower()
                        # if arg_lower.endswith("street") or arg_lower.endswith("avenue"):
                        #     normalized = "road"
                        # else:
                        normalized = arg
                        args.add(normalized)

        # build one sequence by joining all strings
        combined = list(actions) + list(args)
        text = " ".join(combined)

        # tokenize (adds [CLS] and [SEP] by default)
        encoded = tokenizer(text, add_special_tokens=True)
        total_tokens = len(encoded["input_ids"])

        # output
        print(f"\n=== {file_path.name} ===")
        print(f"  • unique action_canon count: {len(actions)}")
        print(f"  • unique args_canon count (after normalization): {len(args)}")
        print(f"  • BERT token length (incl. [CLS]/[SEP]): {total_tokens}")

if __name__ == "__main__":
    DATA_DIR = "/home/will.english/Desktop/Research/AAAI_2026_GinSign/raw_tl_data/VLTL-Bench/total"
    compute_bert_lengths(DATA_DIR)


In [ ]:
import re
from collections import defaultdict
import random

dirs = ["north", "south", "east", "west", "northwest", "northeast", "southwest", "southeast"]
nums = ["1st", "2nd", "3rd", "4th", "5th", "6th", "7th", "8th", "9th", "10th"]
roads = ["street", "avenue"]
all_roads = [dir+"_"+num+"_"+road for dir in dirs for num in nums for road in roads]

# ---------------------------------------------------------------------
# 1.  Signatures
# ---------------------------------------------------------------------
TYPE_CONSTANTS = {
    "<search_and_rescue>": {
        "person": [
            "safe_civilian","safe_hostile","safe_person","safe_rescuer","safe_victim",
            "unsafe_civilian","unsafe_person","unsafe_rescuer","unsafe_victim",
            "injured_civilian","injured_hostile","injured_person","injured_rescuer",
            "injured_victim"
        ],
        "threat": [
            "active_debris","active_fire_source","active_flood","active_gas_leak",
            "active_unstable_beam","debris","fire_source","flood","gas_leak",
            "impending_debris","impending_fire_source","impending_flood",
            "impending_gas_leak","impending_unstable_beam","inactive_debris",
            "inactive_fire_source","inactive_flood","inactive_gas_leak",
            "inactive_unstable_beam","unstable_beam","nearest_fire_source",
            "nearest_flood","nearest_gas_leak","nearest_unstable_beam",
            "probable_debris","probable_fire_source","probable_flood",
            "probable_gas_leak","probable_unstable_beam","nearest_debris"
        ],
    },
    "<traffic_light>": {
        "light": [
            "light_east","light_north","light_south","light_west"
        ],
        "color": ["green","red","yellow"],
        "lane":   # truncated list – add the full list if you need 100 % coverage
            all_roads
        ,
        "traffic_target": [
            "car","collision","cyclist","motorcycle","pedestrian","person",
            "vehicle","jaywalker"
        ],
    },
    "<warehouse>": {
        "item": [
            "aeroplane","apple","backpack","banana","baseball_bat","baseball_glove",
            "bear","bed","bench","bicycle","bird","boat","book","bottle","bowl",
            "broccoli","bus","cake","car","carrot","cat","cell_phone","chair",
            "clock","cow","cup","dining_table","dog","donut","elephant",
            "fire_hydrant","fork","frisbee","giraffe","hair-drier","handbag",
            "horse","hot_dog","keyboard","kite","knife","laptop","microwave",
            "motorbike","mouse","orange","oven","parking_meter","person",
            "pizza","potted_plant","refrigerator","remote","sandwich","scissors",
            "sheep","sink","skateboard","skis","snowboard","sofa","spoon",
            "sports_ball","stop_sign","suitcase","surfboard","teddy-bear",
            "tennis_racket","tie","toaster","toilet","toothbrush","traffic_light",
            "train","truck","tv_monitor","umbrella","vase","wine_glass","zebra"
        ],
        "location": ["loading_dock","shelf"],
        "ego" : []
    },
}

# Every constant split into a bag‑of‑words for cheap matching
BOW = defaultdict(dict)
for dom, tmap in TYPE_CONSTANTS.items():
    for t, consts in tmap.items():
        for c in consts:
            BOW[dom][c] = set(c.split("_"))

STOP_WORDS = {"the", "a", "an", "to", "of"}  # quick‑and‑dirty

TOK_RE = re.compile(r"\w+|[^\w\s]")


import clip
import torch
import torch.nn.functional as F

# ─── 1.  CLIP setup ─────────────────────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model, _ = clip.load("ViT-B/16", device=device)
clip_model.eval()

# similarity threshold for “likely the same”
ITEM_SIM_THRESH = 0.815


# ─── 2.  choose_constant with special item handling ─────────────────────────────
def choose_constant(domain_tag, type_name, sent_tokens, *, predicate=None):   # ← NEW arg
    # 2.a  skip ego entirely
    if type_name == "ego":
        return None

    if type_name == "item":
        tokens = sent_tokens

        def find_match(phrases):
            # best above‐threshold
            best_const      = None
            best_sim        = ITEM_SIM_THRESH
            best_phrase     = None
            matched_consts  = set()

            # global best (any sim)
            global_best_const  = None
            global_best_sim    = float("-inf")
            global_best_phrase = None

            for phrase in phrases:
                query_text = "a photo of a " + " ".join(phrase)
                q_input    = clip.tokenize([query_text]).to(device)
                with torch.no_grad():
                    q_emb = clip_model.encode_text(q_input)

                for const in TYPE_CONSTANTS[domain_tag]["item"]:
                    const_text  = const.replace("_", " ")
                    # const_query = "a photo of a " + const_text
                    const_query = (
                                            "a photo of a "
                                            + (predicate + " " if predicate else "")    # ← NEW
                                            + const_text
                                        )
                    # c_input     = clip.tokenize([const_query]).to(device)

                    c_input     = clip.tokenize([const_query]).to(device)
                    with torch.no_grad():
                        c_emb = clip_model.encode_text(c_input)

                    sim = F.cosine_similarity(q_emb, c_emb).item()

                    # keep track of any that exceed the threshold
                    if sim > ITEM_SIM_THRESH:
                        matched_consts.add(const)

                    # best above threshold?
                    if sim > best_sim:
                        best_sim    = sim
                        best_const  = const
                        best_phrase = phrase

                    # global best?
                    if sim > global_best_sim:
                        global_best_sim    = sim
                        global_best_const  = const
                        global_best_phrase = phrase

            # if nothing ever beat the threshold, fall back to the global best
            if best_const is None and global_best_const is not None:
                best_const  = global_best_const
                best_sim    = global_best_sim
                best_phrase = global_best_phrase

            # still suppress 'person' if it wasn't the *only* above-threshold match
            if best_const == "person" and (matched_consts - {"person"}):
                return None

            # report & return
            qp = " ".join(best_phrase)
            print(f"a photo of a {qp} -> {best_const}  (sim={best_sim:.3f})")
            return best_const


            # run once across *all* candidates
        return find_match([tokens])
            
        # 2.c  otherwise fall back to your previous bag-of-words matcher
        tokens = {t.lower() for t in sent_tokens if t.lower() not in STOP_WORDS}
        best, best_len = None, 0

        for const, bow in BOW[domain_tag].items():
            if const not in TYPE_CONSTANTS[domain_tag].get(type_name, []):
                continue
            # exact cover & prefer longest
            if bow <= tokens and len(bow) > best_len:
                best, best_len = const, len(bow)

        return best

    # 2.c  otherwise fall back to your previous bag-of-words matcher
    tokens = {t.lower() for t in sent_tokens if t.lower() not in STOP_WORDS}
    best, best_len = None, 0

    for const, bow in BOW[domain_tag].items():
        if const not in TYPE_CONSTANTS[domain_tag].get(type_name, []):
            continue
        # exact cover & prefer longest
        if bow <= tokens and len(bow) > best_len:
            best, best_len = const, len(bow)

    return best

# ---------------------------------------------------------------------
# 3.  Main entry point
# ---------------------------------------------------------------------
def build_predicted_target(sentence, prefix, predicted_prefix_target):
    """Return list of strings like ['person:unsafe_civilian', 'record']"""
    domain_tag = prefix[0]              # e.g. '<search_and_rescue>'
    # indices of the special markers in the prefix
    try:
        types_start = prefix.index("<types>") + 1
        preds_start = prefix.index("<predicates>") + 1
    except ValueError:
        raise ValueError("Prefix missing <types> / <predicates> markers.")
    
    types = prefix[types_start : preds_start - 1]
    predicates = prefix[preds_start:]
    out = []
    predicate_tok = None
    for idx, bit in enumerate(predicted_prefix_target):
        if bit and idx >= preds_start:
            predicate_tok = prefix[idx]   # there is exactly one by assumption
            break
    for idx, bit in enumerate(predicted_prefix_target):
        if not bit:
            continue
        tok = prefix[idx]
        #  a)  type  prediction
        if types_start <= idx < preds_start - 1:
            const = choose_constant(
                domain_tag,
                tok,
                sentence,
                predicate=predicate_tok,     # ← pass it along
            )
            if const:
                out.append(f"{tok}:{const}")
            else:               # fall back – at least return the type
                out.append(tok)
        #  b)  predicate prediction
        elif idx >= preds_start:
            out.append(tok)
        #  (ignore special tokens and domain tag)

    return out

# ---------------------------------------------------------------------
# 4.  DEMO
# ---------------------------------------------------------------------
example = {
    "sentence": ["record", "unsafe", "civilian"],
    "prefix": ["<search_and_rescue>", "<types>", "person", "threat", "ego",
               "<predicates>", "avoid", "photo", "record", "deliver_aid",
               "communicate", "go_home", "get_help"],
    "predicted_prefix_target": [0,0,1,0,0,0,0,0,1,0,0,0,0],
}
print(build_predicted_target(**example))



In [ ]:
import json
from pathlib import Path

# Path to your directory
IN_DIR = Path("/home/will.english/Desktop/Research/AAAI_2026_GinSign/grounding_evaluations/bert_only")

# Directory where we’ll write the “naive” outputs
OUT_DIR = IN_DIR.parent / "bert+lexical+clip_embed_pfx_v3"
OUT_DIR.mkdir(exist_ok=True)

for in_path in IN_DIR.glob("*.jsonl"):
    # mirror the filename, but write into grounding_evaluations/naive
    out_path = OUT_DIR / f"{in_path.stem}.jsonl"

    with in_path.open("r", encoding="utf-8") as f_in, \
        out_path.open("w", encoding="utf-8") as f_out:

        for line in f_in:
            data = json.loads(line)
            sent = data.get("sentence", [])
            prefix = data.get("prefix", [])
            pred_flags = data.get("predicted_prefix_target", [])
            # collect all prefix tokens where the corresponding flag == 1
            predicted_target = build_predicted_target(sent, prefix, pred_flags)#[tok for tok, flag in zip(prefix, pred_flags) if flag == 1]
            data["predicted_target"] = predicted_target
            f_out.write(json.dumps(data) + "\n")

    print(f"Processed {in_path.name} → {out_path.name}")


In [ ]:
import json
from pathlib import Path

# Directory containing the original .jsonl files
IN_DIR = Path("/home/will.english/Desktop/Research/AAAI_2026_GinSign/grounding_evaluations/full")

# Directory where we’ll write the “naive” outputs
OUT_DIR = IN_DIR.parent / "bert"
OUT_DIR.mkdir(exist_ok=True)

for in_path in IN_DIR.glob("*.jsonl"):
    # mirror the filename, but write into grounding_evaluations/naive
    out_path = OUT_DIR / f"{in_path.stem}.jsonl"

    with in_path.open("r", encoding="utf-8") as f_in, \
         out_path.open("w", encoding="utf-8") as f_out:

        for line in f_in:
            data = json.loads(line)
            sent = data.get("sentence", [])
            prefix = data.get("prefix", [])

            # create a flag list of all 1s, same length as prefix
            pred_flags = data.get("predicted_prefix_target", [])
            # collect all prefix tokens where the corresponding flag == 1
            predicted_target = [tok for tok, flag in zip(prefix[1:], pred_flags[1:]) if flag == 1]#build_predicted_target(sent, prefix, pred_flags)#[tok for tok, flag in zip(prefix, pred_flags) if flag == 1]
            # data["predicted_target"] = predicted_target
            # assumes build_predicted_target() is defined and in scope
            data["predicted_target"] = predicted_target
            # build_predicted_target(
            #     sentence=sent,
            #     prefix=prefix,
            #     predicted_prefix_target=predicted_target
            # )

            f_out.write(json.dumps(data) + "\n")

    print(f"Processed {in_path.name} → {out_path.relative_to(OUT_DIR.parent)}")


In [ ]:
import json
from pathlib import Path

# Directory containing the original .jsonl files
IN_DIR = Path("/home/will.english/Desktop/Research/AAAI_2026_GinSign/grounding_evaluations/full")

# Directory where we’ll write the “naive” outputs
OUT_DIR = IN_DIR.parent / "naive"
OUT_DIR.mkdir(exist_ok=True)

for in_path in IN_DIR.glob("*.jsonl"):
    # mirror the filename, but write into grounding_evaluations/naive
    out_path = OUT_DIR / f"{in_path.stem}_naive.jsonl"

    with in_path.open("r", encoding="utf-8") as f_in, \
         out_path.open("w", encoding="utf-8") as f_out:

        for line in f_in:
            data = json.loads(line)
            sent = data.get("sentence", [])
            prefix = data.get("prefix", [])

            # create a flag list of all 1s, same length as prefix
            naive_flags = [1] * len(prefix)

            # assumes build_predicted_target() is defined and in scope
            data["predicted_target"] = build_predicted_target(
                sentence=sent,
                prefix=prefix,
                predicted_prefix_target=naive_flags
            )

            f_out.write(json.dumps(data) + "\n")

    print(f"Processed {in_path.name} → {out_path.relative_to(OUT_DIR.parent)}")


In [ ]:
# import json
# from pathlib import Path

# # Define directories
# INPUT_DIR = Path("grounding_evaluations")
# BERT_ONLY_DIR = INPUT_DIR / "bert_only"
# OUTPUT_DIR = Path("processed_llm_grounding")

# bert_info_map = {}
# for bert_file in BERT_ONLY_DIR.glob("*.jsonl"):
#     domain = bert_file.stem  # e.g. "search_and_rescue"
#     with bert_file.open("r", encoding="utf-8") as f:
#         for line in f:
#             rec = json.loads(line)
#             # normalize id to int if possible
#             key_id = rec["id"]
#             try:
#                 key_id = int(key_id)
#             except (ValueError, TypeError):
#                 pass
#             prop = rec["prop_id"]
#             bert_info_map[(domain, key_id, prop)] = {
#                 "target":       rec.get("target", []),
#                 "prefix":       rec.get("prefix", []),
#                 "prefix_target":rec.get("prefix_target", [])
#             }
# # 2) Process only the "3_5_turbo_fine", "4_1_mini_fine" subdirs
# OUTPUT_DIR.mkdir(exist_ok=True)
# for subdir in INPUT_DIR.iterdir():
#     if not subdir.is_dir() or not subdir.name.startswith(("3_5_turbo_fine", "4_1_mini_fine")):
#         continue

#     out_sub = OUTPUT_DIR / subdir.name
#     out_sub.mkdir(exist_ok=True)

#     for src in subdir.glob("*.jsonl"):
#         dst = out_sub / src.name
#         print(src)
#         with src.open() as f_in, dst.open("w") as f_out:
#             for line in f_in:
#                 rec = json.loads(line)
#                 # parse out domain and numeric id
#                 custom = rec.get("custom_id","")
#                 domain = custom.split("-",1)[0]
#                 try:
#                     id_val = int(custom.split("-")[-1])
#                 except ValueError:
#                     print("error")
#                     id_val = custom

#                 # parse the LLM’s JSON response
#                 choices = rec.get("response",{}).get("body",{}).get("choices",[])
#                 if not choices: 
#                     continue
#                 content = choices[0]["message"].get("content","")
#                 try:
#                     preds = json.loads(content)
#                 except json.JSONDecodeError:
#                     print("error")
                    

#                 # for each prop_n, build a new entry
#                 for prop_id, info in preds.items():
#                     try:
#                         action = info.get("action_canon")
#                     except Exception:
#                         action = ""
#                     try:
#                         args = info.get("args_canon", [])
#                     except Exception:
#                         args = []

#                     sentence = ([action] if action else []) + args
#                     predicted_target = sentence[:]   # action + args

#                     # pull in the bert-only gold info
#                     bert_key = (domain, id_val, prop_id)
#                     bert_info = bert_info_map.get(bert_key, {})
#                     true_target       = bert_info.get("target", [])
#                     true_prefix       = bert_info.get("prefix", [])
#                     true_prefix_targs = bert_info.get("prefix_target", [])

#                     # recompute predicted_prefix_target against the gold prefix
#                     predicted_prefix_targs = [
#                         1 if tok in predicted_target else 0
#                         for tok in true_prefix
#                     ]

#                     new_entry = {
#                         "id":                     id_val,
#                         "prop_id":                prop_id,
#                         "sentence":               sentence,
#                         "target":                 true_target,
#                         "prefix":                 true_prefix,
#                         "prefix_target":          true_prefix_targs,
#                         "predicted_prefix_target":predicted_prefix_targs,
#                         "predicted_target":       predicted_target,
#                     }
#                     f_out.write(json.dumps(new_entry) + "\n")
import json
from pathlib import Path

# 1) load the gold info
INPUT_DIR     = Path("grounding_evaluations")
BERT_ONLY_DIR = INPUT_DIR / "bert_only"
OUTPUT_DIR    = Path("processed_llm_grounding")
bert_info_map = {}

for bert_file in BERT_ONLY_DIR.glob("*.jsonl"):
    domain = bert_file.stem
    with bert_file.open("r", encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            key_id = rec["id"]
            try:
                key_id = int(key_id)
            except (ValueError, TypeError):
                pass
            prop_id = rec["prop_id"]
            bert_info_map[(domain, key_id, prop_id)] = {
                "target":        rec.get("target", []),
                "prefix":        rec.get("prefix", []),
                "prefix_target": rec.get("prefix_target", [])
            }

# 2) process only the two subdirs you want
OUTPUT_DIR.mkdir(exist_ok=True)
for subdir in INPUT_DIR.iterdir():
    if not subdir.is_dir() or not subdir.name.startswith(("3_5_turbo_fine", "4_1_mini_fine")):
        continue

    out_sub = OUTPUT_DIR / subdir.name
    out_sub.mkdir(exist_ok=True)

    for src in subdir.glob("*.jsonl"):
        dst = out_sub / src.name
        print(f"→ processing {src}")
        with src.open("r", encoding="utf-8") as f_in, dst.open("w", encoding="utf-8") as f_out:
            for line in f_in:
                rec = json.loads(line)

                # extract domain & numeric id
                custom = rec.get("custom_id", "")
                domain = custom.split("-", 1)[0]
                try:
                    id_val = int(custom.split("-")[-1])
                except ValueError:
                    print(f"  ! bad custom_id: {custom!r}")
                    continue

                # parse the LLM’s JSON response (robust to extra text or code fences)
                choices = rec.get("response", {}) \
                             .get("body", {}) \
                             .get("choices", [])
                if not choices:
                    continue
                raw = choices[0]["message"].get("content", "").strip()

                # strip markdown code fences if present
                if raw.startswith("```"):
                    lines = raw.splitlines()
                    # drop the first and last fence lines
                    if len(lines) > 2 and lines[-1].startswith("```"):
                        raw = "\n".join(lines[1:-1])

                # find the outermost JSON braces
                start = raw.find("{")
                end   = raw.rfind("}")
                if start < 0 or end < 0 or end <= start:
                    print(f"  ! can't locate JSON braces for id={id_val}")
                    continue

                json_str = raw[start:end+1]
                try:
                    preds = json.loads(json_str)
                except json.JSONDecodeError as e:
                    print(f"  ! JSON decode failed for id={id_val}: {e}")
                    continue

                # one output row per prop
                for prop_id, info in preds.items():
                    action = info.get("action_canon", "") or ""
                    args   = info.get("args_canon", []) or []
                    sentence = ([action] if action else []) + args
                    predicted_target = sentence[:]   # simple action+args

                    # look up the gold
                    key = (domain, id_val, prop_id)
                    if key not in bert_info_map:
                        print(f"  ! MISSING GOLD for {key}")
                        continue    # skip if no gold entry

                    gold = bert_info_map[key]
                    true_target       = gold["target"]
                    true_prefix       = gold["prefix"]
                    true_prefix_targs = gold["prefix_target"]

                    # rebuild your prefix mask
                    predicted_prefix_targs = [
                        1 if tok in predicted_target else 0
                        for tok in true_prefix
                    ]

                    new_entry = {
                        "id":                       id_val,
                        "prop_id":                  prop_id,
                        "sentence":                 sentence,
                        "target":                   true_target,
                        "prefix":                   true_prefix,
                        "prefix_target":            true_prefix_targs,
                        "predicted_prefix_target":  predicted_prefix_targs,
                        "predicted_target":         predicted_target,
                    }
                    f_out.write(json.dumps(new_entry, ensure_ascii=False) + "\n")


In [ ]:
import json, re
from pathlib import Path

# ---------------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------------
ID_RE = re.compile(r"(\d+)$")          # grab trailing integer once
def extract_id(custom_id: str) -> int | str:
    m = ID_RE.search(custom_id)
    return int(m.group(1)) if m else custom_id          # fallback: raw string

def infer_types_section(prefix: list[str]) -> list[str]:
    """Return the list of <types> tokens that appear between <types> and <predicates>."""
    if "<types>" not in prefix:
        return []
    start = prefix.index("<types>") + 1
    try:
        end = prefix.index("<predicates>")
    except ValueError:
        end = len(prefix)
    return prefix[start:end]

def constant_to_type(constant: str, type_tokens: list[str]) -> str | None:
    """
    Try to map a constant name (e.g. injured_rescuer) to one of the type tokens
    seen in the gold prefix, falling back to a few heuristics.
    """
    # 1) direct match / prefix / suffix w.r.t. the gold tokens
    for t in type_tokens:
        if constant.startswith(t + "_") or constant.endswith("_" + t) or constant.startswith(t):
            return t

    # 2) quick heuristics that work well for the SAR & traffic-light domains
    PERSON_WORDS = ("rescuer", "victim", "civilian", "person", "hostile")
    THREAT_WORDS = ("flood", "debris", "fire", "smoke", "obstacle")
    if constant.endswith(PERSON_WORDS):
        return "person"
    if any(word in constant for word in THREAT_WORDS):
        return "threat"
    if "light" in constant:
        return "light"
    if constant.endswith(("street", "avenue")):
        return "road"
    return None

def build_predicted_target(action: str | None, args: list[str], type_tokens: list[str]) -> list[str]:
    out: list[str] = []
    for arg in args:
        t = constant_to_type(arg, type_tokens)
        token = f"{t}:{arg}" if t else arg      # keep extra detail like "threat:flood"
        if token not in out:
            out.append(token)
    if action and action not in out:
        out.append(action)
    return out

def prefix_hit(tok: str, predicted: list[str]) -> bool:
    """Is this prefix token covered by ANY predicted token?"""
    for p in predicted:
        if p == tok or p.startswith(tok + ":"):
            return True
    return False

# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------
INPUT_DIR      = Path("grounding_evaluations")
BERT_ONLY_DIR  = INPUT_DIR / "bert_only"
OUTPUT_DIR     = Path("processed_llm_grounding")

# ---------------------------------------------------------------------
# 1) load the gold BERT-only information
# ---------------------------------------------------------------------
bert_info_map = {}
for bert_file in BERT_ONLY_DIR.glob("*.jsonl"):
    domain = bert_file.stem           # e.g. search_and_rescue
    with bert_file.open(encoding="utf-8") as f:
        for line in f:
            rec     = json.loads(line)
            key_id  = int(rec["id"]) if isinstance(rec["id"], (int, str)) and str(rec["id"]).isdigit() else rec["id"]
            prop_id = rec["prop_id"]
            bert_info_map[(domain, key_id, prop_id)] = {
                "target":        rec.get("target", []),
                "prefix":        rec.get("prefix", []),
                "prefix_target": rec.get("prefix_target", [])
            }

# ---------------------------------------------------------------------
# 2) walk every model output directory we care about
# ---------------------------------------------------------------------
OUTPUT_DIR.mkdir(exist_ok=True)
for subdir in INPUT_DIR.iterdir():
    if not subdir.is_dir() or not subdir.name.startswith(("3_5", "4_1", "4o")):
        continue

    out_sub = OUTPUT_DIR / subdir.name
    out_sub.mkdir(exist_ok=True)

    for src in subdir.glob("*.jsonl"):
        dst = out_sub / src.name
        print(f"⇢ {src.relative_to(INPUT_DIR)}  →  {dst.relative_to(OUTPUT_DIR)}")

        with src.open(encoding="utf-8") as f_in, dst.open("w", encoding="utf-8") as f_out:
            for raw in f_in:
                rec = json.loads(raw)

                custom      = rec.get("custom_id", "")
                domain      = custom.split("-", 1)[0]           # takes text before *first* '-'
                id_val      = extract_id(custom)

                # ----- parse the LLM JSON response -----
                try:
                    choices = rec["response"]["body"]["choices"]
                    content = choices[0]["message"]["content"]
                    preds   = json.loads(content)
                except (KeyError, IndexError, json.JSONDecodeError, TypeError):
                    # skip bad / empty / non-JSON answers
                    continue

                # write a new line PER prop_n
                for prop_id, info in preds.items():
                    action = info.get("action_canon", "")
                    args   = info.get("args_canon", []) or []

                    # gold lookup
                    bert_key  = (domain, id_val, prop_id)
                    bert_info = bert_info_map.get(bert_key)
                    if not bert_info:
                        # skip if we don't have gold data – nothing to compare against
                        continue

                    prefix        = bert_info["prefix"]
                    type_tokens   = infer_types_section(prefix)
                    predicted     = build_predicted_target(action, args, type_tokens)

                    prefix_hits   = [
                        1 if prefix_hit(tok, predicted) else 0
                        for tok in prefix
                    ]

                    new_entry = {
                        "id":                       id_val,
                        "prop_id":                  prop_id,
                        "sentence":                 ([action] if action else []) + args,
                        "target":                   bert_info["target"],
                        "prefix":                   prefix,
                        "prefix_target":            bert_info["prefix_target"],
                        "predicted_prefix_target":  prefix_hits,
                        "predicted_target":         predicted,
                    }
                    f_out.write(json.dumps(new_entry, ensure_ascii=False) + "\n")


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Evaluate grounding accuracy for every METHOD directory under
`grounding_evaluations/`, reporting scores per domain (dataset) and overall.
"""
from pathlib import Path
import json, collections

ROOT = Path("grounding_evaluations")        # top-level folder
METHOD_DIRS = [p for p in ROOT.iterdir() if p.is_dir()]

# --------------------------------------------------------------------------- #
# 1.  DOMAIN SIGNATURES – import or paste your TYPE_CONSTANTS dict here
# --------------------------------------------------------------------------- #

# --------------------------------------------------------------------------- #
# 2.  helper utilities
# --------------------------------------------------------------------------- #
def split_prefix(prefix):
    dom_tag = prefix[0]                      # e.g. "<warehouse>"
    t0      = prefix.index("<types>")      + 1
    p0      = prefix.index("<predicates>") + 1
    return dom_tag, set(prefix[t0:p0-1]), set(prefix[p0:])

def const_to_type(domain_tag, const):
    for t, consts in TYPE_CONSTANTS[domain_tag].items():
        if const in consts:
            return t
    return None

def categorise(tokens, domain_tag, type_vocab, pred_vocab):
    types, preds, consts = set(), set(), set()
    for tok in tokens:
        if ":" in tok:           # type:constant form
            t, c = tok.split(":", 1)
            types.add(t); consts.add(c)
        elif tok in type_vocab: types.add(tok)
        elif tok in pred_vocab:  preds.add(tok)
        else:                    # assume bare constant
            consts.add(tok)
            t = const_to_type(domain_tag, tok)
            if t: types.add(t)
    return types, preds, consts

def update_ctr(pred, gold, ctr):
    ctr["tp"] += len(pred & gold)
    ctr["fp"] += len(pred - gold)
    ctr["fn"] += len(gold - pred)

def prf(ctr):
    p = ctr["tp"] / (ctr["tp"] + ctr["fp"]) if ctr["tp"] + ctr["fp"] else 0.0
    r = ctr["tp"] / (ctr["tp"] + ctr["fn"]) if ctr["tp"] + ctr["fn"] else 0.0
    f = 2*p*r/(p+r) if p+r else 0.0
    return p, r, f

def new_ctr():
    return collections.Counter(tp=0, fp=0, fn=0)

# --------------------------------------------------------------------------- #
# 3.  evaluation loop
# --------------------------------------------------------------------------- #
for method in sorted(METHOD_DIRS):
    overall = {cat: new_ctr() for cat in ("type", "predicate", "constant")}
    domain_ctrs = collections.defaultdict(
        lambda: {cat: new_ctr() for cat in ("type", "predicate", "constant")})

    # ---------------- iterate result files ---------------- #
    for jsonl in method.glob("*.jsonl"):
        domain = jsonl.stem  # traffic_light, warehouse, ...
        with jsonl.open("r", encoding="utf-8") as f:
            for line in f:
                obj = json.loads(line)
                tgt, pred, prefix = obj["target"], obj["predicted_target"], obj["prefix"]
                # print(obj["prefix"])
                dom_tag, type_vocab, pred_vocab = split_prefix(prefix)

                tgt_T, tgt_P, tgt_C   = categorise(tgt,  dom_tag, type_vocab, pred_vocab)
                pred_T, pred_P, pred_C = categorise(pred, dom_tag, type_vocab, pred_vocab)

                for cat, (p, g) in zip(
                    ("type","predicate","constant"),
                    ((pred_T,tgt_T),(pred_P,tgt_P),(pred_C,tgt_C))):
                    update_ctr(p, g, overall[cat])
                    update_ctr(p, g, domain_ctrs[domain][cat])

    # ---------------- print summary ----------------------- #
    print(f"\n====== {method.name} ======")
    for dom, ctrs in domain_ctrs.items():
        print(f"\n  --- {dom} ---")
        for cat in ("type","predicate","constant"):
            p,r,f = prf(ctrs[cat])
            print(f"  {cat:10s} P={p:.3f} R={r:.3f} F1={f:.3f}")

    print("\n  === overall ===")
    for cat in ("type","predicate","constant"):
        p,r,f = prf(overall[cat])
        print(f"  {cat:10s} P={p:.3f} R={r:.3f} F1={f:.3f}")


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Evaluate grounding accuracy for every METHOD directory under
`processed_llm_grounding/`, reporting scores per domain and overall.
"""

from pathlib import Path
import json, collections

# --------------------------------------------------------------------------- #
# 0.  where the processed predictions live
# --------------------------------------------------------------------------- #
ROOT = Path("grounding_evaluations")       # <— changed
METHOD_DIRS = [p for p in ROOT.iterdir() if p.is_dir()]

# --------------------------------------------------------------------------- #
# 1.  DOMAIN SIGNATURES – import or paste your TYPE_CONSTANTS dict here
# --------------------------------------------------------------------------- #
TYPE_CONSTANTS = { ... }     # ← your existing dict goes here

# --------------------------------------------------------------------------- #
# 2.  helper utilities
# --------------------------------------------------------------------------- #
def split_prefix(prefix):
    dom_tag = prefix[0]                      # e.g. "<warehouse>"
    t0      = prefix.index("<types>")      + 1
    p0      = prefix.index("<predicates>") + 1
    return dom_tag, set(prefix[t0:p0-1]), set(prefix[p0:])

def const_to_type(domain_tag, const):
    for t, consts in TYPE_CONSTANTS[domain_tag].items():
        if const in consts:
            return t
    return None

def categorise(tokens, domain_tag, type_vocab, pred_vocab):
    types, preds, consts = set(), set(), set()
    for tok in tokens:
        if ":" in tok:           # type:constant form
            t, c = tok.split(":", 1)
            types.add(t); consts.add(c)
        elif tok in type_vocab:  types.add(tok)
        elif tok in pred_vocab:  preds.add(tok)
        else:                    # assume bare constant
            consts.add(tok)
            t = const_to_type(domain_tag, tok)
            if t: types.add(t)
    return types, preds, consts

def update_ctr(pred, gold, ctr):
    ctr["tp"] += len(pred & gold)
    ctr["fp"] += len(pred - gold)
    ctr["fn"] += len(gold - pred)

def prf(ctr):
    p = ctr["tp"] / (ctr["tp"] + ctr["fp"]) if ctr["tp"] + ctr["fp"] else 0.0
    r = ctr["tp"] / (ctr["tp"] + ctr["fn"]) if ctr["tp"] + ctr["fn"] else 0.0
    f = 2*p*r/(p+r) if p+r else 0.0
    return p, r, f

def new_ctr():
    return collections.Counter(tp=0, fp=0, fn=0)

# --------------------------------------------------------------------------- #
# 3.  evaluation loop
# --------------------------------------------------------------------------- #
for method in sorted(METHOD_DIRS):
    overall = {cat: new_ctr() for cat in ("type", "predicate", "constant")}
    domain_ctrs = collections.defaultdict(
        lambda: {cat: new_ctr() for cat in ("type", "predicate", "constant")})

    # ---------------- iterate result files ---------------- #
    for jsonl in method.glob("*.jsonl"):
        domain = jsonl.stem
        with jsonl.open("r", encoding="utf-8") as f:
            for line in f:
                obj = json.loads(line)
                # skip malformed / gold-only lines
                if not all(k in obj for k in ("target", "predicted_target", "prefix")):
                    continue

                tgt, pred, prefix = obj["target"], obj["predicted_target"], obj["prefix"]
                dom_tag, type_vocab, pred_vocab = split_prefix(prefix)

                tgt_T, tgt_P, tgt_C   = categorise(tgt,  dom_tag, type_vocab, pred_vocab)
                pred_T, pred_P, pred_C = categorise(pred, dom_tag, type_vocab, pred_vocab)

                for cat, (p, g) in zip(
                        ("type","predicate","constant"),
                        ((pred_T,tgt_T),(pred_P,tgt_P),(pred_C,tgt_C))):
                    update_ctr(p, g, overall[cat])
                    update_ctr(p, g, domain_ctrs[domain][cat])

    # ---------------- print summary ----------------------- #
    print(f"\n====== {method.name} ======")
    for dom, ctrs in domain_ctrs.items():
        print(f"\n  --- {dom} ---")
        for cat in ("type","predicate","constant"):
            p,r,f = prf(ctrs[cat])
            print(f"  {cat:10s} P={p:.3f} R={r:.3f} F1={f:.3f}")

    print("\n  === overall ===")
    for cat in ("type","predicate","constant"):
        p,r,f = prf(overall[cat])
        print(f"  {cat:10s} P={p:.3f} R={r:.3f} F1={f:.3f}")


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Evaluate grounding accuracy for every METHOD directory under
`grounding_evaluations/`, reporting P/R/F1 and Exact-Match per domain
and overall.
"""

import re
from collections import defaultdict
import random

dirs = ["north", "south", "east", "west", "northwest", "northeast", "southwest", "southeast"]
nums = ["1st", "2nd", "3rd", "4th", "5th", "6th", "7th", "8th", "9th", "10th"]
roads = ["street", "avenue"]
all_roads = [dir+"_"+num+"_"+road for dir in dirs for num in nums for road in roads]

# ---------------------------------------------------------------------
# 1.  Signatures
# ---------------------------------------------------------------------
TYPE_CONSTANTS = {
    "<search_and_rescue>": {
        "person": [
            "safe_civilian","safe_hostile","safe_person","safe_rescuer","safe_victim",
            "unsafe_civilian","unsafe_person","unsafe_rescuer","unsafe_victim",
            "injured_civilian","injured_hostile","injured_person","injured_rescuer",
            "injured_victim"
        ],
        "threat": [
            "active_debris","active_fire_source","active_flood","active_gas_leak",
            "active_unstable_beam","debris","fire_source","flood","gas_leak",
            "impending_debris","impending_fire_source","impending_flood",
            "impending_gas_leak","impending_unstable_beam","inactive_debris",
            "inactive_fire_source","inactive_flood","inactive_gas_leak",
            "inactive_unstable_beam","unstable_beam","nearest_fire_source",
            "nearest_flood","nearest_gas_leak","nearest_unstable_beam",
            "probable_debris","probable_fire_source","probable_flood",
            "probable_gas_leak","probable_unstable_beam","nearest_debris"
        ],
    },
    "<traffic_light>": {
        "light": [
            "light_east","light_north","light_south","light_west"
        ],
        "color": ["green","red","yellow"],
        "lane":   # truncated list – add the full list if you need 100 % coverage
            all_roads
        ,
        "traffic_target": [
            "car","collision","cyclist","motorcycle","pedestrian","person",
            "vehicle","jaywalker"
        ],
    },
    "<warehouse>": {
        "item": [
            "aeroplane","apple","backpack","banana","baseball_bat","baseball_glove",
            "bear","bed","bench","bicycle","bird","boat","book","bottle","bowl",
            "broccoli","bus","cake","car","carrot","cat","cell_phone","chair",
            "clock","cow","cup","dining_table","dog","donut","elephant",
            "fire_hydrant","fork","frisbee","giraffe","hair-drier","handbag",
            "horse","hot_dog","keyboard","kite","knife","laptop","microwave",
            "motorbike","mouse","orange","oven","parking_meter","person",
            "pizza","potted_plant","refrigerator","remote","sandwich","scissors",
            "sheep","sink","skateboard","skis","snowboard","sofa","spoon",
            "sports_ball","stop_sign","suitcase","surfboard","teddy-bear",
            "tennis_racket","tie","toaster","toilet","toothbrush","traffic_light",
            "train","truck","tv_monitor","umbrella","vase","wine_glass","zebra"
        ],
        "location": ["loading_dock","shelf"],
        "ego" : []
    },
}

from pathlib import Path
import json, collections

ROOT = Path("grounding_evaluations")        # top-level folder
METHOD_DIRS = [p for p in ROOT.iterdir() if p.is_dir()]

# --------------------------------------------------------------------------- #
# 1.  TYPE_CONSTANTS dict should be imported or pasted here
# --------------------------------------------------------------------------- #

# --------------------------------------------------------------------------- #
# 2.  helpers
# --------------------------------------------------------------------------- #
def split_prefix(prefix):
    raw     = prefix[0]             # "<search_and_rescue>"
    dom_tag = raw.strip("<>")       # "search_and_rescue"
    t0      = prefix.index("<types>")      + 1
    p0      = prefix.index("<predicates>") + 1
    return dom_tag, set(prefix[t0:p0-1]), set(prefix[p0:])

def const_to_type(domain_tag, const):
    mapping = TYPE_CONSTANTS.get(domain_tag)
    if not isinstance(mapping, dict):
        return None
    for t, consts in mapping.items():
        if const in consts:
            return t
    return None


def categorise(tokens, domain_tag, type_vocab, pred_vocab):
    # 1) bail if the whole thing isn’t a list
    if not isinstance(tokens, list):
        return set(), set(), set()

    types, preds, consts = set(), set(), set()
    for tok in tokens:
        # 2) skip any None or non‐str
        if not isinstance(tok, str):
            continue

        if ":" in tok:
            t, c = tok.split(":", 1)
            types.add(t)
            consts.add(c)
        elif tok in type_vocab:
            types.add(tok)
        elif tok in pred_vocab:
            preds.add(tok)
        else:
            consts.add(tok)
            t = const_to_type(domain_tag, tok)
            if t:
                types.add(t)

    return types, preds, consts


def update_ctr(pred, gold, ctr):
    ctr["tp"] += len(pred & gold)
    ctr["fp"] += len(pred - gold)
    ctr["fn"] += len(gold - pred)

def prf(ctr):
    p = ctr["tp"] / (ctr["tp"] + ctr["fp"]) if ctr["tp"] + ctr["fp"] else 0.0
    r = ctr["tp"] / (ctr["tp"] + ctr["fn"]) if ctr["tp"] + ctr["fn"] else 0.0
    f = 2 * p * r / (p + r) if (p + r) else 0.0
    return p, r, f

def new_ctr():
    return collections.Counter(tp=0, fp=0, fn=0)

def new_em():
    return collections.Counter(match=0, total=0)

metrics = {}

# --------------------------------------------------------------------------- #
# 3.  evaluation loop
# --------------------------------------------------------------------------- #
for method in sorted(METHOD_DIRS):
    overall_prf = {c: new_ctr() for c in ("type", "predicate", "constant")}
    overall_em  = {c: new_em()  for c in ("type", "predicate", "constant")}

    domain_prf = collections.defaultdict(
        lambda: {c: new_ctr() for c in ("type", "predicate", "constant")})
    domain_em = collections.defaultdict(
        lambda: {c: new_em() for c in ("type", "predicate", "constant")})

    # ---------------- iterate result files ---------------- #
    for jsonl in method.glob("*.jsonl"):
        domain = jsonl.stem
        with jsonl.open(encoding="utf-8") as f:
            for line in f:
                obj = json.loads(line)

                # 1) safely grab your three fields (fall back to empty lists)
                gold = obj.get("target") if isinstance(obj.get("target"), list) else []
                pred = obj.get("predicted_target") if isinstance(obj.get("predicted_target"), list) else []
                prefix = obj.get("prefix")

                # 2) you must have a prefix list to split off your domain/types/preds
                if not isinstance(prefix, list) or not prefix:
                    print(f"  ! skipping line with no valid prefix: {line.strip()}")
                    continue

                dom_tag, type_vocab, pred_vocab = split_prefix(prefix)

                # 3) now safe to categorize
                gT, gP, gC = categorise(gold, dom_tag, type_vocab, pred_vocab)
                pT, pP, pC = categorise(pred, dom_tag, type_vocab, pred_vocab)

                for cat, (p, g) in zip(
                    ("type", "predicate", "constant"),
                    ((pT, gT), (pP, gP), (pC, gC))
                ):
                    update_ctr(p, g, overall_prf[cat])
                    update_ctr(p, g, domain_prf[domain][cat])

                    overall_em[cat]["total"] += 1
                    domain_em[domain][cat]["total"] += 1
                    if p == g:
                        overall_em[cat]["match"] += 1
                        domain_em[domain][cat]["match"] += 1

    # metrics.insert(method.name)

    method_metrics = collections.defaultdict(dict)   # per-domain store

    # ---------------- iterate result files ---------------- #
    for jsonl in method.glob("*.jsonl"):
        domain = jsonl.stem
        # … (existing loop that fills counters) …

    # ---------------- compute & save all scores ----------- #
    for dom in domain_prf:
        domain_dict = {}
        for cat in ("type", "predicate", "constant"):
            p, r, f = prf(domain_prf[dom][cat])
            em = domain_em[dom][cat]["match"] / domain_em[dom][cat]["total"]
            domain_dict[cat] = {"P": p, "R": r, "F1": f, "EM": em}
        method_metrics[dom] = domain_dict

    metrics[method.name] = method_metrics        # <- store for plotting

    # ---------------- print summary (unchanged) ------------ #
    print(f"\n====== {method.name} ======")
    for dom in domain_prf:
        print(f"\n  --- {dom} ---")
        for cat in ("type", "predicate", "constant"):
            d = method_metrics[dom][cat]
            print(f"  {cat:10s}  P={d['P']:.3f} R={d['R']:.3f} "
                  f"F1={d['F1']:.3f}  EM={d['EM']:.3f}")

    print("\n  === overall ===")
    for cat in ("type", "predicate", "constant"):
        p, r, f = prf(overall_prf[cat])
        em = overall_em[cat]["match"] / overall_em[cat]["total"]
        print(f"  {cat:10s}  P={p:.3f} R={r:.3f} F1={f:.3f}  EM={em:.3f}")


In [ ]:
import json, pathlib
with Path("grounding_metrics.json").open("w") as fp:
    json.dump(metrics, fp, indent=2)

In [ ]:
print(metrics)
methods = sorted(metrics.keys(), key=lambda m: (m != "lexical", m))

print(methods)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Assume 'metrics' dict is already in the namespace

# 1. Order methods
methods = sorted(metrics.keys(), key=lambda m: (m != "lexical", m))
# In case 'lexical' isn't actually a key, remove that lambda or adjust as needed:
# methods = sorted(metrics.keys())

domains = ["traffic_light", "search_and_rescue", "warehouse"]
domain_labels = {
    "traffic_light": "Traffic Light",
    "search_and_rescue": "Search & Rescue",
    "warehouse": "Warehouse",
}

def draw_layered_bars(values_by_domain, title, ylabel):
    x = np.arange(len(methods))
    bar_w = 0.8
    plt.figure(figsize=(10, 4))
    for dom in domains:
        plt.bar(
            x,
            values_by_domain.get(dom, [0]*len(methods)),
            width=bar_w,
            alpha=0.6,
            label=domain_labels[dom]
        )
    plt.xticks(x, methods, rotation=45, ha="right")
    plt.ylim(0, 1.05)
    plt.title(title)
    plt.ylabel(ylabel)
    plt.legend()
    plt.tight_layout()
    plt.show()


# Coarse-grained: Type & Predicate
for metric_key, pretty in [("F1", "F1"), ("EM", "Exact Match")]:
    for cat_key, cat_name in [("type", "Type"), ("predicate", "Predicate")]:
        vals = {
            dom: [
                metrics.get(m, {}).get(dom, {}).get(cat_key, {}).get(metric_key, 0.0)
                for m in methods
            ]
            for dom in domains
        }
        draw_layered_bars(
            vals,
            title=f"Coarse-grained Grounding – {cat_name} {pretty}",
            ylabel=pretty
        )

# Fine-grained: Constant
for metric_key, pretty in [("F1", "F1"), ("EM", "Exact Match")]:
    vals = {
        dom: [
            metrics.get(m, {}).get(dom, {}).get("constant", {}).get(metric_key, 0.0)
            for m in methods
        ]
        for dom in domains
    }
    draw_layered_bars(
        vals,
        title=f"Fine-grained Grounding – Constant {pretty}",
        ylabel=pretty
    )


In [ ]:

import os
import glob
import json
import numpy as np
import pandas as pd
import re
from utils import parse_temporal_logic
from pyModelChecking.LTL import Parser
parser = Parser()
import spot

def levenshtein(seq1, seq2):
    """
    Compute the Levenshtein distance between two sequences (lists of tokens).
    """
    m, n = len(seq1), len(seq2)
    # dp[i][j] = edit distance between seq1[:i] and seq2[:j]
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        dp[i][0] = i
    for j in range(1, n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if seq1[i-1] == seq2[j-1] else 1
            dp[i][j] = min(
                dp[i-1][j] + 1,      # deletion
                dp[i][j-1] + 1,      # insertion
                dp[i-1][j-1] + cost  # substitution
            )
    return dp[m][n]
def wrap_prop_quotes(tokens):
    """
    Given a list of TL tokens, wrap any prop_N in quotes.
    """
    return [
        f'"{t}"' if re.fullmatch(r'prop_\d+', t) else t
        for t in tokens
    ]

def compute_metrics_for_file(path):
    entries = [json.loads(l) for l in open(path, 'r', encoding='utf-8')]
    entries = entries[:500]
    n = len(entries)

    # 1) Binary accuracy
    correct = [ (e['prediction'] == e['masked_tl']) for e in entries ]
    binary_acc = sum(correct) / n

    # 2) Levenshtein distances
    dists = [ levenshtein(e['prediction'], e['masked_tl']) for e in entries ]
    mean = np.mean(dists)
    var  = np.var(dists, ddof=1)
    se   = np.std(dists, ddof=1) / np.sqrt(n)
    ci95 = 1.96 * se
    eqs = []
    parse_errors = 0

    for idx, e in enumerate(entries):
        if len(eqs) >= 500:
            break
        # 1) turn token lists into strings
        msk_tokens = wrap_prop_quotes(e['masked_tl'])
        msk_str    = parse_temporal_logic(" ".join(msk_tokens))

        # 2. Predicted TL
        if isinstance(e['prediction'], list):
            pred_tokens = wrap_prop_quotes(e['prediction'])

        else:
            pred_tokens = wrap_prop_quotes(e['prediction'].split(" "))
        pred_str    = parse_temporal_logic(" ".join(pred_tokens))

        try:
            # 3) attempt to build formula objects
            m_form = parser(msk_str)
            p_form = parser(pred_str)
            eqs.append(m_form == p_form)

        except Exception as ex:
            # DEBUG OUTPUT: everything you need to diagnose
            print("────────── PARSE FAILURE ──────────")
            print(f"Entry index: {idx}, id: {e.get('id')}")
            print(" Raw masked_tl tokens:")
            print("   ", e['masked_tl'])
            print(" Raw prediction tokens:")
            print("   ", e['prediction'])
            print(" After parse_temporal_logic & arrow‐fix:")
            print("   masked_str =", repr(msk_str))
            print("   pred_str   =", repr(pred_str))
            print(" Parser error:", ex, "\n")
            # Re-raise so you get the full traceback on *this* string
            # raise

            eqs.append(pred_str == msk_str)

    logical_eq_acc = sum(eqs) / n

    return {
        'binary_acc':     binary_acc,
        'lev_mean':       mean,
        'lev_variance':   var,
        'lev_ci95_low':   mean - ci95,
        'lev_ci95_high':  mean + ci95,
        'logical_eq_acc': logical_eq_acc,
        'parse_errors':   parse_errors,
        'total':          n
    }


root_dir='translation_evaluations' 
results = []
for method in os.listdir(root_dir):
    method_dir = os.path.join(root_dir, method)
    if not os.path.isdir(method_dir):
        continue
    for filepath in glob.glob(os.path.join(method_dir, '*.jsonl')):
        dataset = os.path.splitext(os.path.basename(filepath))[0]
        metrics = compute_metrics_for_file(filepath)
        metrics.update({'method': method, 'dataset': dataset})
        results.append(metrics)

# assemble into DataFrame for easy viewing / CSV export
df = pd.DataFrame(results)
print(df)
df.to_csv('translation_metrics_summary.csv', index=False)
print("\nWritten summary to translation_metrics_summary.csv")



In [ ]:
import json
from pathlib import Path

def load_grounding_predictions(grounding_dir):
    """
    Load all grounding-evaluation entries into a dict keyed by (domain, id, prop_id).
    Each value is a dict with:
      - "target": the gold target list
      - "predicted": the predicted_target list
    """
    grounding_map = {}
    for fp in Path(grounding_dir).glob("*.jsonl"):
        domain = fp.stem
        with fp.open(encoding="utf-8") as f:
            for line in f:
                rec = json.loads(line)
                key = (domain, rec["id"], rec["prop_id"])
                grounding_map[key] = {
                    "target": rec.get("target", []),
                    "predicted": rec.get("predicted_target", [])
                }
    return grounding_map

def strip_predicted(pred_list):
    """
    For each entry like "person:safe_hostile" → keep only "safe_hostile".
    Entries without a colon stay unchanged.
    """
    return [s.split(":", 1)[-1] for s in pred_list]

def is_example_correct(rec, domain, grounding_map):
    """
    Returns True if:
      1) rec['prediction'] exactly matches rec['masked_tl'], AND
      2) for every prop_i in masked_tl, the stripped predicted_target
         matches the gold target (as sets).
    """
    # 1) structural check
    if rec["prediction"] != rec["masked_tl"]:
        return False

    # 2) grounding check
    props = {tok for tok in rec["masked_tl"] if tok.startswith("prop_")}
    for prop in props:
        key = (domain, rec["id"], prop)
        grd = grounding_map.get(key)
        if grd is None:
            # missing grounding entry
            return False

        gold_target = grd["target"]
        pred_stripped = strip_predicted(grd["predicted"])

        # compare as sets so order doesn't matter
        if set(pred_stripped) != set(gold_target):
            return False

    return True

def evaluate(domains, trans_dir, grounding_dir):
    grounding_map = load_grounding_predictions(grounding_dir)
    results = {}

    for domain in domains:
        total    = 0
        correct  = 0
        mismatches = []

        trans_fp = Path(trans_dir) / f"{domain}.jsonl"
        with trans_fp.open(encoding="utf-8") as f:
            for line in f:
                rec = json.loads(line)
                total += 1

                if is_example_correct(rec, domain, grounding_map):
                    correct += 1
                else:
                    mismatches.append({
                        "id": rec["id"],
                        "prediction": rec["prediction"],
                        "masked_tl": rec["masked_tl"]
                    })

        accuracy = correct / total if total else 0.0
        results[domain] = {
            "total": total,
            "correct": correct,
            "accuracy": accuracy,
            "mismatches": mismatches
        }

    return results

if __name__ == "__main__":
    domains       = ["search_and_rescue", "warehouse", "traffic_light"]
    trans_dir     = "translation_evaluations/GinSign"
    grounding_dir = "grounding_evaluations/bert+lexical+clip_embed_pfx_v3"

    metrics = evaluate(domains, trans_dir, grounding_dir)

    for domain, stats in metrics.items():
        print(f"Domain: {domain}")
        print(f"  Total examples : {stats['total']}")
        print(f"  Correct matches: {stats['correct']}")
        print(f"  Accuracy       : {stats['accuracy']:.2%}")
        if stats["mismatches"]:
            print("  Sample mismatches:")
            for e in stats["mismatches"][:5]:
                print(f"    id={e['id']}, pred={e['prediction']}, masked={e['masked_tl']}")
        print()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Read the summary CSV
df = pd.read_csv('translation_metrics_summary.csv')

# Print a well-formatted table
print(df.to_markdown(index=False))

# Plot key metrics for each method across datasets
metrics = ['binary_acc', 'lev_mean', 'logical_eq_acc']
for metric in metrics:
    plt.figure()
    for method in df['method'].unique():
        subset = df[df['method'] == method]
        plt.plot(subset['dataset'], subset[metric], marker='o', label=method)
    plt.title(metric.replace('_', ' ').title())
    plt.xlabel('Dataset')
    plt.ylabel(metric.replace('_', ' ').title())
    plt.legend()
    plt.tight_layout()
    plt.show()
